# Capítulo 8 — Engenharia de dados para IA

## Missão
Montar um **pipeline de dados de treino e de avaliação** de qualidade, partindo de dados sujos. Ele vai:
1. inspecionar;
2. tirar duplicatas exatas e quase iguais (com MinHash feito à mão);
3. limpar e filtrar (inclusive dados pessoais);
4. medir a cobertura;
5. **gerar dados sintéticos** com o modelo onde faltam e verificar esses dados com um juiz;
6. formatar para treino.

Dados bons valem mais que um modelo maior.

**No fim deste caderno você terá:** o módulo `meu_assistente/dados_treino.py` (pipeline limpar → deduplicar → filtrar → formatar) e um conjunto final com relatório de cada etapa.

**Leitura essencial (opcional, livro em inglês):**
- Data Curation: Quality, Coverage, Quantity · PDF p. 389
- AI-Powered Data Synthesis · PDF p. 410
- Model Distillation · PDF p. 419
- Data Processing: Inspect, Deduplicate, Clean, Format · PDF p. 420

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap08_engenharia_de_dados_para_ia.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
Três perguntas guiam a curadoria:
- **qualidade**: os exemplos estão certos?
- **cobertura**: eles cobrem os casos reais?
- **quantidade**: são suficientes?

Dados sintéticos (gerados por um modelo) tapam buracos de cobertura, mas precisam de verificação, porque o modelo também erra e repete padrões.

Dados sujos de propósito (já vem pronto): pares pergunta/resposta com duplicatas, quase duplicatas, vazios, inglês, textos longos demais e dados pessoais.

In [ ]:
sujos = [{"pergunta": p["pergunta"], "resposta": p["referencia"], "doc": p["doc"]} for p in PERGUNTAS]
sujos += [
    {"pergunta": "Quanto custa o plano Equipe?", "resposta": "R$ 29 por usuário por mês.", "doc": "planos"},
    {"pergunta": "quanto custa o plano equipe??", "resposta": "R$ 29 por usuário por mês.", "doc": "planos"},
    {"pergunta": "Quanto custa o plano Equipe por mês?", "resposta": "R$ 29 por usuário/mês.", "doc": "planos"},
    {"pergunta": "", "resposta": "Sem pergunta.", "doc": "planos"},
    {"pergunta": "How do I reset my password?", "resposta": "Click on forgot password.", "doc": "senha"},
    {"pergunta": "Meu e-mail é joana.silva@gmail.com e meu CPF 123.456.789-00, não consigo entrar.",
     "resposta": "Use Esqueci minha senha.", "doc": "senha"},
    {"pergunta": "Explique tudo sobre a Nimbus " + "por favor " * 80, "resposta": "...", "doc": "planos"},
]
df = pd.DataFrame(sujos)
print(len(df), "linhas")

### 8.1 **Inspeção.** Mostre: número de linhas, quantas perguntas vazias, a distribuição do tamanho das perguntas em palavras (`describe`) e as 3 maiores. Inspecionar antes de tudo evita surpresas.

### 8.2 **Duplicatas exatas.** Normalize as perguntas (minúsculas, sem acento, sem pontuação, espaços únicos), calcule um hash (`hashlib.md5`) e remova as repetidas. Quantas saíram?

### 8.3 **Quase duplicatas com MinHash.** Implemente:
1. `shingles(t, k=3)`: o conjunto de trincas de palavras do texto normalizado;
2. a Jaccard exata;
3. um MinHash com 64 funções de hash (`hash((semente, s))`): a assinatura é o mínimo de cada função, e a estimativa da Jaccard é a fração de posições iguais.

Compare a Jaccard exata com a estimada para dois pares de perguntas e marque como quase duplicatas os pares com estimativa >= 0,5.

### 8.4 **Limpeza e filtros.** Escreva regras e aplique em `sem_exatas`:
- tirar perguntas vazias ou com mais de 60 palavras;
- tirar as que parecem inglês (heurística: palavras como `the`, `how`, `my`, `do`);
- **mascarar** e-mails e CPFs com regex, trocando por `[EMAIL]` e `[CPF]`.

Mostre quantas linhas sobraram e a pergunta mascarada.

### 8.5 **Cobertura.** Conte quantas perguntas há por documento (`doc`) e liste os documentos da Nimbus com **1 pergunta ou menos**. São esses os buracos que os dados sintéticos vão tapar.

### 8.6 **Aumento tradicional (sem IA).** Gere variações de cada pergunta trocando palavras por sinônimos de um dicionário pequeno (por exemplo, `quanto custa` → `qual o preço de`, `consigo` → `é possível`). Quantas perguntas novas saíram?

### 8.7 **Dados sintéticos com IA.** Para cada documento com poucas perguntas, peça ao modelo 2 pares pergunta/resposta baseados **só** no texto do documento (JSON `{"itens": [{"pergunta": ..., "resposta": ...}]}`). Guarde em `sinteticos` com o `doc` de origem.

### 8.8 **Verificação com juiz.** Para cada par sintético, pergunte ao `MODELO_FORTE` se a resposta está correta e é sustentada pelo documento (JSON `{"relevante": true/false, "motivo": ...}`). Mantenha só os aprovados. Quantos passaram?

### 8.9 **Módulo do pipeline.** Escreva `dados_treino.py` com `preparar_conjunto(pares, sistema)`, que recebe uma lista de dicts (`pergunta`, `resposta`) e faz, contando o que sai em cada etapa:
1. tira vazios;
2. tira duplicatas exatas;
3. tira perguntas com mais de 60 palavras;
4. mascara e-mail e CPF;
5. formata no padrão de conversa (`messages`).

Devolve `(exemplos, relatorio)`, em que `relatorio` é um dict etapa → quantidade.

_A célula grava `meu_assistente/dados_treino.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/dados_treino.py


### 8.10 **Rode o pipeline completo.** Junte os dados sujos, as variações por sinônimo e os sintéticos aprovados, passe por `preparar_conjunto` e grave `../saidas/conjunto_final.jsonl`. Mostre o relatório como tabela.

### 8.11 **Destilação.** Ideia: o modelo forte gera as respostas e um modelo menor aprende com elas. Para 3 perguntas dos `TICKETS`, gere a resposta de suporte com `MODELO_FORTE` ("professor") e monte os exemplos de treino do "aluno" no formato `messages`. Quanto isso custou (`gasto()`)?

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
